# Asynchronous Deep Double Duelling Q-Learning for Trading-Signal Execution in Limit Order Book Markets
**ArXivist-generated reproduction notebook**

Paper: [arXiv:2301.08688](https://arxiv.org/abs/2301.08688)  
Authors: Peer Nagy, Jan-Peter Calliess, Stefan Zohren (Oxford-Man Institute, 2023)  
Generated: 2026-06-07

This notebook walks through the key components of the implementation, runs a small-scale training loop on synthetic data, and verifies the setup matches the paper's structure. Estimated runtime: **< 10 minutes** on a modern GPU, **< 30 minutes** on CPU.

In [ ]:
# Cell 1 — Environment Check
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be slower but fully functional")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

In [ ]:
# Cell 2 — Install project in editable mode (run once)
import subprocess, sys
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".."],
        capture_output=True, text=True, cwd=".."
    )
    if result.returncode == 0:
        print("Installation successful.")
    else:
        print("Installation error:", result.stderr[:500])
except Exception as e:
    print(f"Could not install automatically: {e}")
    print("Run manually: pip install -e .. (from repo root)")

# Add src to path as fallback
import sys, pathlib
sys.path.insert(0, str(pathlib.Path("..").resolve() / "src"))

## Paper Overview

### What problem does it solve?

Quantitative trading strategies typically work in two stages: (1) generate a **directional signal** predicting short-term price moves, then (2) **execute** that signal — deciding exactly when to place orders and at what price. Stage 2 is often hand-crafted. This paper automates it with deep RL.

### Core idea

Train an RL agent in a **realistic limit order book (LOB) simulator** (ABIDES + LOBSTER data) to convert a noisy directional signal into individual limit orders. The agent learns:
- **When** to trade (vs. skip)
- **Where** to place orders: at the bid (passive), mid-price, or ask (aggressive)
- **How** to manage inventory between $-10$ and $+10$ shares

### Key innovation

- **7-action space** including passive limit order placement (unlike prior work limited to market orders)
- **APEX architecture**: 42 parallel CPU workers collect experience asynchronously; 1 GPU learner trains
- **Curriculum reward**: starts with directional signal reward, transitions to pure P&L
- **First application of APEX to LOB environments**

### How code maps to paper

| Paper Section | Code File |
|---|---|
| §3.1 LOB data | `data/lob_dataset.py` |
| §3.2 APEX DQN | `models/q_network.py`, `training/trainer.py` |
| §4.1 Signal | `data/signal_generator.py` |
| §4.2 RL env | `training/environment.py` |
| §5 Baseline | `evaluation/baseline.py` |
| §5 Metrics | `evaluation/metrics.py` |

## Component 1: Synthetic Directional Signal (Section 4.1, Eq. 1–2)

The agent receives a noisy oracle signal of the mean return $h=10$ seconds ahead:

$$d_t = \phi d_{t-1} + (1-\phi)\epsilon_t, \quad \epsilon_t \sim \text{Dirichlet}(\alpha(r_{t+h}))$$

$$\alpha(r_{t+h}) = \begin{cases}(a^H, a^L, a^L) & r < -k \\ (a^L, a^H, a^L) & |r| < k \\ (a^L, a^L, a^H) & r \geq k\end{cases}$$

$d_t \in \Delta^2$ is a 3-class probability vector: **[p_down, p_neutral, p_up]**. Higher $a^H$ → better signal quality.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from apex_lob_trader.data.signal_generator import SignalGenerator

# Demonstrate signal at three noise levels (matching paper's a values)
np.random.seed(42)
T = 500
# Synthetic mid-price random walk
mid_prices = 570.0 * np.exp(np.cumsum(np.random.normal(0, 5e-5, T + 20)))

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, a_H, label in zip(axes, [1.6, 1.3, 1.1], ["a=1.6 (low noise)", "a=1.3 (mid)", "a=1.1 (high noise)"]):
    sg = SignalGenerator(a_H=a_H, a_L=1.0, phi=0.9, horizon_h=10, threshold_k=4e-5)
    up_probs = []
    for t in range(T):
        d = sg.step(mid_prices, t)
        up_probs.append(d[2])  # p_up
    ax.plot(up_probs, lw=0.8, color='steelblue')
    ax.axhline(0.5, color='gray', lw=0.6, ls='--')
    ax.set_ylabel("P(up)")
    ax.set_title(label)
    ax.set_ylim(0, 1)
axes[-1].set_xlabel("Time step")
plt.suptitle("Directional Signal (Section 4.1) — P(up) at Three Noise Levels", fontsize=13)
plt.tight_layout()
plt.show()
print(f"SignalGenerator: {sg}")

## Component 2: Duelling Q-Network (Section 3.2, [26][27])

Architecture:
- **Input**: $[B, L, D]$ where $L=100$ history steps, $D=10$ state features per step
- **3 FF layers** applied per time step → **LSTM** over the 100-step sequence
- **Value stream** $V(s)$: scalar — how good is this state?
- **Advantage stream** $A(s,a)$: per-action — how much better is action $a$?
- **Q-value**: $Q(s,a) = V(s) + A(s,a) - \text{mean}_a A(s,a)$

In [ ]:
import torch
from apex_lob_trader.models.q_network import DuellingQNetwork

# Instantiate with paper config
model = DuellingQNetwork(
    state_dim=10,
    history_len=100,
    hidden_dim=256,    # ASSUMED — see SIR IA-01
    num_actions=7,
    num_ff_layers=3,
).to(device)

# Demo forward pass with a batch of 2
B, L, D = 2, 100, 10
try:
    x = torch.randn(B, L, D).to(device)
    q_values, hidden = model(x)
    print(f"Input shape:   {x.shape}")
    print(f"Q-values shape: {q_values.shape}  (expected: [2, 7])")
    print(f"Sample Q-values (batch 0): {q_values[0].detach().cpu().numpy().round(3)}")
    print(f"\nModel: {model}")
    print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
except Exception as e:
    print(f"Error in forward pass: {e}")

## Component 3: LOB Trading Environment (Section 4.2)

**Action space** (7 discrete actions):

| Index | Action | Description |
|---|---|---|
| 0 | sell @ bid | Passive sell (resting limit order) |
| 1 | sell @ mid | Mid-price sell |
| 2 | sell @ ask | Aggressive sell (crosses spread) |
| 3 | buy @ bid | Aggressive buy (crosses spread) |
| 4 | buy @ mid | Mid-price buy |
| 5 | buy @ ask | Passive buy (resting limit order) |
| 6 | skip | Do nothing |

**Reward** (Eq. 3):
$$r_t = w^{dir} R^{dir}_t + (1 - w^{dir}) R^{pnl}_t, \quad w^{dir} \leftarrow \psi w^{dir}$$

In [ ]:
from apex_lob_trader.data.lob_dataset import LOBDataset
from apex_lob_trader.training.environment import LOBTradingEnv
import yaml

# Load config
try:
    with open("../configs/config.yaml") as f:
        cfg = yaml.safe_load(f)
except FileNotFoundError:
    print("Config not found — using inline mini-config for demo")
    cfg = {
        "env": {
            "history_len": 20,           # reduced for demo speed
            "state_dim_per_step": 10,
            "inventory": {"pos_min": -10, "pos_max": 10},
            "data": {"data_dir": "../data/lobster/", "asset": "AAPL"}
        },
        "signal": {"noise_level_a": 1.3, "a_L": 1.0, "persistence_phi": 0.9, "horizon_h": 10, "return_threshold_k": 4e-5},
        "training": {"reward": {"w_dir_initial": 0.5, "psi_decay": 0.9999, "kappa": 1.0}}
    }

# Use reduced history for demo
cfg["env"]["history_len"] = 20

try:
    dataset = LOBDataset(
        data_dir=cfg["env"]["data"]["data_dir"],
        asset=cfg["env"]["data"]["asset"],
        split="train",
        history_len=cfg["env"]["history_len"],
    )
    dataset.load()
    env = LOBTradingEnv(dataset=dataset, cfg=cfg)
    print(f"Environment: {env}")
    print(f"Observation space: {env.observation_space}")
    print(f"Action space: {env.action_space}")

    # Test a few random steps
    obs, _ = env.reset()
    print(f"\nInitial obs shape: {obs.shape}")
    for step_i in range(5):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        print(f"  Step {step_i+1}: action={action}, reward={reward:.5f}, inventory={info['inventory']}")
except Exception as e:
    print(f"Environment error: {e}")

## Component 4: Replay Buffer (Section 3.2)

The paper **disables prioritised replay** (footnote 1): uniform sampling is used instead. Buffer size = 2,000,000 transitions.

In [ ]:
from apex_lob_trader.training.replay_buffer import ReplayBuffer, Transition
import numpy as np

obs_dim = cfg["env"]["history_len"] * cfg["env"]["state_dim_per_step"]

try:
    buf = ReplayBuffer(capacity=10000, obs_dim=obs_dim, seed=42)  # small for demo
    # Push 200 random transitions
    for _ in range(200):
        t = Transition(
            state=np.random.randn(obs_dim).astype(np.float32),
            action=np.random.randint(7),
            reward=float(np.random.randn()),
            next_state=np.random.randn(obs_dim).astype(np.float32),
            done=bool(np.random.random() < 0.05),
        )
        buf.push(t)

    states, actions, rewards, next_states, dones = buf.sample(32)
    print(f"Buffer: {buf}")
    print(f"Sampled batch — states: {states.shape}, actions: {actions.shape}, rewards: {rewards[:5].round(3)}")
except Exception as e:
    print(f"Replay buffer error: {e}")

## Mini-Training Loop

Running a short training loop (200 steps) on **synthetic data** to verify the full pipeline. This is NOT sufficient to reproduce paper results — use `train.py` with 300M steps and real LOBSTER data for that.

In [ ]:
# Mini-config for fast demo training
mini_cfg = {
    "seed": 42,
    "env": {
        "history_len": 10,
        "state_dim_per_step": 10,
        "inventory": {"pos_min": -10, "pos_max": 10},
        "data": {"data_dir": "../data/lobster/", "asset": "AAPL"}
    },
    "signal": {"noise_level_a": 1.3, "a_L": 1.0, "persistence_phi": 0.9,
               "horizon_h": 10, "return_threshold_k": 4e-5},
    "model": {"num_ff_layers": 3, "hidden_dim": 64, "num_actions": 7},  # smaller for speed
    "training": {
        "total_timesteps": 200,
        "gamma": 0.99,
        "replay_buffer": {"size": 5000, "prioritized": False},
        "learning": {
            "learning_starts": 50,
            "train_batch_size": 32,
            "rollout_fragment_length": 50,
            "target_update_freq": 100,
            "n_step": 3,
        },
        "lr_schedule": [[0, 2e-5], [1000000, 5e-6]],
        "reward": {"w_dir_initial": 0.5, "psi_decay": 0.9999, "kappa": 1.0},
    },
    "hardware": {"device": "cpu", "deterministic": False},
    "logging": {"log_every_n_steps": 50, "checkpoint_every_n_steps": 99999,
                "checkpoint_dir": "/tmp/apex_lob_ckpts", "results_dir": "/tmp/apex_lob_results"}
}

In [ ]:
import torch
from apex_lob_trader.data.lob_dataset import LOBDataset
from apex_lob_trader.training.environment import LOBTradingEnv
from apex_lob_trader.training.trainer import APEXDQNTrainer

try:
    mini_dataset = LOBDataset(
        data_dir=mini_cfg["env"]["data"]["data_dir"],
        asset="AAPL", split="train",
        history_len=mini_cfg["env"]["history_len"],
    )
    mini_dataset.load()

    mini_env = LOBTradingEnv(dataset=mini_dataset, cfg=mini_cfg)
    mini_trainer = APEXDQNTrainer(env=mini_env, cfg=mini_cfg, device=torch.device("cpu"))

    print(f"Mini model params: {sum(p.numel() for p in mini_trainer.main_net.parameters()):,}")
    print("Running 200-step debug training...\n")
    mini_trainer.train(debug=True)

    if mini_trainer.losses:
        import matplotlib.pyplot as plt
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(mini_trainer.losses)
        ax1.set_title("Training Loss (last 200 steps)")
        ax1.set_xlabel("Update step")
        ax1.set_ylabel("Smooth L1 Loss")
        if mini_trainer.episode_returns:
            ax2.plot(mini_trainer.episode_returns)
            ax2.set_title("Episode Returns")
            ax2.set_xlabel("Episode")
            ax2.set_ylabel("Cumulative log-return")
        plt.tight_layout()
        plt.show()
        print(f"\nLoss converging: first={mini_trainer.losses[0]:.4f}, last={mini_trainer.losses[-1]:.4f}")
except Exception as e:
    print(f"Mini-training error: {e}")
    import traceback; traceback.print_exc()

## Component 5: Heuristic Baseline (Section 5)

The baseline trades **aggressively** in the signal direction and **passively** unwinds when neutral.

In [ ]:
from apex_lob_trader.evaluation.baseline import HeuristicBaseline
import numpy as np

try:
    baseline = HeuristicBaseline(pos_min=-10, pos_max=10)

    # Test all signal scenarios
    scenarios = [
        (np.array([0.7, 0.2, 0.1]), 5,   "Strong DOWN signal, long inventory"),
        (np.array([0.1, 0.2, 0.7]), -3,  "Strong UP signal, short inventory"),
        (np.array([0.2, 0.6, 0.2]), 3,   "NEUTRAL signal, long inventory → unwind"),
        (np.array([0.2, 0.6, 0.2]), 0,   "NEUTRAL signal, flat → skip"),
    ]

    action_names = ["sell@bid","sell@mid","sell@ask","buy@bid","buy@mid","buy@ask","skip"]
    print(f"{'Scenario':<45} {'Action':>12}")
    print("-" * 60)
    for signal, inv, desc in scenarios:
        action = baseline.act(signal, inv)
        print(f"{desc:<45} {action_names[action]:>12}")
except Exception as e:
    print(f"Baseline error: {e}")

## Paper Results (for reference)

The following reproduces Table / Figure 2 results from Section 5. Run `evaluate.py` with a trained checkpoint to compare your results.

In [ ]:
from apex_lob_trader.evaluation.metrics import mean_log_return, sharpe_ratio, print_results_table

# Results reported in the paper (Section 5, Figure 2)
paper_results = {
    "noise_a": [1.1, 1.3, 1.6],
    "label": ["High noise (a=1.1)", "Mid noise (a=1.3)", "Low noise (a=1.6)"],
    "rl_mu":         [0.00, 0.11, 0.21],
    "rl_sharpe":     [-0.72, 7.34, 14.69],
    "baseline_mu":   [-0.07, 0.00, None],
    "baseline_sharpe": [-5.68, -0.04, None],
    "outperf_pp":    [32.2, 14.8, 20.7],
}

print("Paper reported results (Section 5, 31 test episodes):\n")
print(f"{'Noise Level':<22} {'RL μ':>8} {'RL S':>8} {'Base μ':>8} {'Base S':>8} {'ΔPerf pp':>10}")
print("-" * 70)
for i, label in enumerate(paper_results["label"]):
    bm = paper_results["baseline_mu"][i]
    bs = paper_results["baseline_sharpe"][i]
    print(
        f"{label:<22}"
        f"{paper_results['rl_mu'][i]:>8.2f}"
        f"{paper_results['rl_sharpe'][i]:>8.2f}"
        f"{str(round(bm,2)) if bm is not None else 'N/A':>8}"
        f"{str(round(bs,2)) if bs is not None else 'N/A':>8}"
        f"{paper_results['outperf_pp'][i]:>10.1f}"
    )
print("\nAll RL vs baseline differences are statistically significant (p << 0.1, t-test).")
print("To reproduce: run train.py with 300M steps on LOBSTER AAPL 2012 data.")
print("Then: python evaluate.py --checkpoint checkpoints/final.pt")

## What to do next

1. **Get data**: Obtain LOBSTER AAPL 2012 data (see `data/README_data.md`)
2. **Full training**: `python train.py --config configs/config.yaml` (300M steps, 42 workers via RLlib)
3. **Evaluation**: `python evaluate.py --checkpoint checkpoints/final.pt --noise-level 1.3`
4. **Experiment**: Try different noise levels (`a=1.1`, `1.3`, `1.6`) to reproduce Figure 2

**Key implementation assumptions to verify** (from SIR):
- `hidden_dim=256` — not stated in paper (IA-01, confidence 0.55)
- `w_dir_initial=0.5`, `psi=0.9999` — not stated (IA-02, confidence 0.40)
- `kappa=1.0` — not stated (IA-03, confidence 0.40)

**LOBSTER data**: Commercial dataset — see `data/README_data.md`. Synthetic fallback provided for testing.